# 桶排序
对AED因子进行传统的桶排序，并且构建对冲组合，分析每个桶的收益和夏普。   




## 导入库

In [5]:
import warnings
from typing import Any
from pathlib import Path
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.io as pio
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import statsmodels.api as sm
from scipy.stats import t as t_dist

## 超参数 

In [6]:
import os
import dotenv
dotenv.load_dotenv()
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

BUCKETS_NUM = 10 # 分桶数  

RISK_FREE_RATE = 0.015 / 12 # 无风险利率  

## 读取数据  
需要读取AED数据和市值数据  
市值数据：数据库 or 本地json (测试用)   


In [7]:
aed_lf = pl.scan_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-AED因子.parquet')   
# 市值数据从 DB 读取（statics.market_value）
market_value_lf = pl.read_database_uri(
    uri=CONNECTION_URL,
    query='SELECT stkcd AS "Stkcd", trdmnt AS "Trdmnt", msmvosd AS "Msmvosd" FROM statics.market_value',
    engine=ENGINE,
).lazy()

## 处理数据(测试)

In [8]:
market_value_lf = market_value_lf.rename(
    {
        'Stkcd':'portfolio',
        'Trdmnt':'date',
        'Msmvosd':'msmvosd'
    }
)
aed_lf.head().collect()

date,portfolio,AED,return
date,str,f64,f64
2021-03-01,"""002483""",2.596032,0.0485
2018-07-01,"""300226""",1.341226,-0.1245
2021-03-01,"""600502""",1.824548,-0.0382
2020-10-01,"""002628""",2.59455,0.0077
2015-02-01,"""300251""",1.082017,0.0035


## 桶排序  
按照AED因子对数据进行桶排序，按分位数构建BUCKETS_NUM个组合，组合内部使用市值加权平均计算组合收益，然后绘制各个组合的累计收益。   
同时，还需要构建一个对冲组合，其收益为最后一个组合和第一个组合的差值。     

### 构建桶  
#### 分桶组合
首先进行分桶，获取每个桶的时序收益(series_list)


In [9]:
# 分桶
aed_lf = aed_lf.with_columns(
    [
        pl.col('AED').quantile(i / BUCKETS_NUM, interpolation='lower').alias(f'q{i}')
        for i in range(1,BUCKETS_NUM)
    ]
)

joined_lf = aed_lf.join(market_value_lf, on=['portfolio','date'], how='left')
joined_lf = joined_lf.with_columns(
    (pl.col('msmvosd') / pl.col('msmvosd').sum().over('date')).alias('weight')
)

series_list = list[pl.LazyFrame]() # 每一个桶对应的series，date-weighted_sum_ret
mean_ret_list = []  # 每一个桶的月平均收益  
for i in range(0,BUCKETS_NUM): # 按照AED由小到大排序  
    # 第1个组合
    if i == 0:
        bucket_lf = joined_lf.filter(pl.col(f'AED') <= pl.col(f'q1'))
    elif i == (BUCKETS_NUM - 1):
        bucket_lf = joined_lf.filter(pl.col(f'AED') > pl.col(f'q{i}'))
    else:
        bucket_lf = joined_lf.filter((pl.col(f'AED') > pl.col(f'q{i}')) & (pl.col(f'AED') <= pl.col(f'q{i+1}')))
    bucket_lf = bucket_lf.select(['date','portfolio','weight','return'])
    
    # 获取series
    series_lf = bucket_lf.group_by(['date']).agg(
        (pl.col('weight') * pl.col('return')).sum().alias('weighted_sum_ret'),
        pl.lit(i).alias('bucket_id')
    )

    # 添加到series_list
    series_list.append(series_lf)
   
# 合并
combined_series = pl.concat(series_list, how='vertical').sort(['date','bucket_id'])
combined_series = combined_series.with_columns(pl.col('bucket_id').cast(pl.Utf8)) # 转为字符串，便于添加对冲组合

#### 对冲组合  
做多第BUCKET_NUM-1个组合，做空第1个组合。 

In [10]:
# 构建对冲组合
low_bucket = combined_series.filter(pl.col('bucket_id') == str(0)).rename({'weighted_sum_ret':'low_bucket_ret'})
high_bucket = combined_series.filter(pl.col('bucket_id') == str(BUCKETS_NUM - 1)).rename({'weighted_sum_ret':'high_bucket_ret'})  
joined_bucket = low_bucket.join(high_bucket, on='date', how='left')
joined_bucket = joined_bucket.with_columns(
    pl.col('high_bucket_ret').fill_null(0).alias('high_bucket_ret'),
    pl.col('low_bucket_ret').fill_null(0).alias('low_bucket_ret'),
)
joined_bucket = joined_bucket.with_columns(
    (pl.col('high_bucket_ret') - pl.col('low_bucket_ret')).alias('weighted_sum_ret')
)
joined_bucket = joined_bucket.select('date','weighted_sum_ret').with_columns(
    pl.lit('对冲组合').alias('bucket_id')
)

joined_bucket.head().collect()

date,weighted_sum_ret,bucket_id
date,f64,str
2005-12-01,0.00225,"""对冲组合"""
2006-09-01,0.009419,"""对冲组合"""
2011-05-01,0.004732,"""对冲组合"""
2023-11-01,0.000364,"""对冲组合"""
2005-08-01,0.002724,"""对冲组合"""


In [11]:
combined_series = pl.concat([combined_series, joined_bucket], how='vertical').sort(['date','bucket_id'])
combined_series.head().collect()

date,weighted_sum_ret,bucket_id
date,f64,str
2004-12-01,-0.014113,"""0"""
2004-12-01,-0.006996,"""1"""
2004-12-01,-0.001943,"""2"""
2004-12-01,-0.002173,"""3"""
2004-12-01,-0.000289,"""4"""


In [12]:
if SAVE:
    combined_series.collect().write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶市值加权收益.parquet')

### 平均收益
#### 每个桶的平均收益（包括对冲组合）    
对于每个桶序列，计算其各个时期的平均收益, HAC-t 和 p  

计算方式为，使用`weighted_sum_ret`回归常数项，使用HAC-t和p。  

In [13]:
# 每个 bucket：weighted_sum_ret 的 mean、HAC-t、p（只回归常数项，无 market_ret）

coll = combined_series.collect()
def regress_one(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy()
        x = np.ones((len(y), 1))
        results = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f'计算{bid}时发生错误: {e}')
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [0.0],
            't': [0.0],
            'p': [1.0],
        })
mean_tp_table = coll.group_by('bucket_id').map_groups(regress_one)

In [14]:
# 格式化：4 位有效数字
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
# t、p 加括号
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    pl.col('mean_return'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
# 居中对齐到 12 位
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
# 合并为一行展示
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    (pl.col('mean_return') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('mean_return'),
).sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(mean_tp_table)

bucket_id,mean_return
str,str
"""0""",""" -0.0022 [-3.011] (0.002605) """
"""1""",""" -0.000957 [-2.158] (0.03092) """
"""2""",""" 0.0001727 [0.4647] (0.6421) """
"""3""",""" 0.0005049 [1.41] (0.1586) """
"""4""",""" 0.0005588 [1.814] (0.06962) """
…,…
"""6""",""" 0.000739 [2.425] (0.01529) """
"""7""",""" 0.001259 [3.207] (0.00134) """
"""8""",""" 0.001104 [3.049] (0.002293) """


In [15]:
if SAVE:
    mean_tp_table.write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶均值HACtp.parquet')

将对冲组合的收益序列添加到桶排序的底部  

绘制panel_list的累计收益曲线  

绘制方法为，对于每一个时序数据，计算每一期累计收益，然后绘制成曲线。  

In [16]:
# 累加
combined_series = combined_series.with_columns(pl.col('weighted_sum_ret').cum_sum().over('bucket_id').alias('cum_return'))

# 百分化
combined_series = combined_series.with_columns(
    pl.col('cum_return').mul(100).alias('cum_return')
)

# 平滑
ROLLING_WINDOW = 4
MIN_SAMPLES = 1
combined_series = combined_series.with_columns(
    pl.col('cum_return').rolling_mean(window_size=ROLLING_WINDOW, min_samples=MIN_SAMPLES).over('bucket_id').alias('cum_return_smooth')
)

# 绘图
fig = px.line(combined_series.collect(), x='date', y='cum_return_smooth', color='bucket_id')
fig.update_layout(
    title=f'累计收益按桶分组(平滑窗口={ROLLING_WINDOW},最小样本={MIN_SAMPLES})',           # 图标题
    xaxis_title='日期',                # x 轴名称
    yaxis_title=f'累计收益率(%)',           # y 轴名称
)
fig.show()
    
    

In [17]:
if SAVE:
    fig.write_image(SAVE_BASELINE_REG_DIR + '/基准回归-累计收益率分桶.png')

### Sharp  
计算每个组合的Sharp比率    

计算方式为：$(mean(ret) - RISK\_FREE\_RATE) / std(ret)$


In [18]:
sharp_series = combined_series.group_by(['bucket_id']).agg(
    pl.col('weighted_sum_ret').mean().alias('mean_ret'),
    pl.col('weighted_sum_ret').std().alias('std_ret'),
)

sharp_series = sharp_series.select(
    pl.col('bucket_id'),
    ((pl.col('mean_ret') - RISK_FREE_RATE) / pl.col('std_ret')).alias('sharp')
)

sharp_series = sharp_series.with_columns(
    pl.col('sharp').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('sharp'),
)
sharp_series = sharp_series.sort('bucket_id')
sharp_series = sharp_series.collect()
sharp_series

bucket_id,sharp
str,str
"""0""","""-0.3798"""
"""1""","""-0.3661"""
"""2""","""-0.2068"""
"""3""","""-0.1562"""
"""4""","""-0.1558"""
…,…
"""6""","""-0.1295"""
"""7""","""0.001923"""
"""8""","""-0.03321"""


### 合并+保存 收益和sharp

In [19]:
performance = mean_tp_table.join(sharp_series, on='bucket_id', how='left').sort('bucket_id')

if SAVE:
    performance.write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶表现.parquet')